# ADC analysis (7-bit SAR, sky130)

Reads the files written by the ngspice control blocks and reports linearity, ENOB and THD. It does **not** run simulations.

| File | Written by | Used for |
| --- | --- | --- |
| `adc_transfer.csv` (`vd code eoc idd_ua code_dout`) | `adc_control_all.spice` | gain, offset, INL, DNL, missing codes |
| `adc_transfer_b.csv`, `adc_sine_b.csv` | `adc_control_all.spice` | second runs, to catch simulator glitches |
| `adc_sine.csv` (`n vd code eoc idd_ua code_dout`) | `adc_control_all.spice` | SINAD, ENOB, THD, SFDR |

**`code` and `code_dout`.** `code` is read from the SAR register just before the last edge (plus d_out0), `code_dout` from the d_out register after it. They should be identical; if not, the notebook says which bits are lost and when. Older files without the last column still load.

**What the sine test measures.** Each sample is its own conversion with the input held constant, so the input is ideally sampled and the result is the *static-nonlinearity* ENOB and THD. Effects of the input moving during the conversion (the converter has no sample-and-hold) are not in it.

**ENOB** is `(SINAD - 1.76 + 20 log10(2^N / A_pp)) / 6.02`, where `A_pp` is the measured peak-to-peak signal in codes, so it is corrected for a signal below full scale. The ideal-quantizer row is the same samples quantized by a perfect 7-bit ADC with the same gain and offset: the gap to it is what the converter adds.

**INL and DNL** are limited by how finely the input is swept. Each transition can only be placed to the nearest sweep point, so with one point per LSB the quantization error alone looks like about half an LSB of INL, and it repeats with a period set by how the step beats against the code width. To show this, the notebook runs the same sweep points through a perfect ADC with the measured gain and offset and draws it in grey/black. Only the part of the measured curves beyond that is the circuit. The INL check passes only if the measured value plus that floor is inside the limit, fails only if the measured value minus the floor is outside it, and otherwise reports INFO (use a finer sweep: 4 or more points per LSB, `npts` of 513 or more).

In [ ]:
from pathlib import Path

DATA_DIR = Path('simulation/adc.sch/')

N_BITS   = 7
VDD      = 1.8
R_OHM    = 31.5e3            # the V-to-I resistor (for the ideal gain only; use the measured value if you have it)
I_FS     = 20e-6             # ladder full-scale current
F_CONV   = 1e6               # conversions per second

RAMP_FILE = 'adc_transfer.csv'       # adc_control_all.spice: vd code eoc idd_ua code_dout
SINE_FILE = 'adc_sine.csv'           # adc_control_all.spice: n vd code eoc idd_ua code_dout
RAMP_FILE_B = 'adc_transfer_b.csv'   # second run of the same sweep (adc_control_all.spice, different max time step): catches simulator glitches
SINE_FILE_B = 'adc_sine_b.csv'       # second run of the sine (adc_control_all.spice)
SINE_N   = 128                       # samples and cycles-per-record in the sine run (must match `sn`, `sm` in adc_control_all.spice)
SINE_M   = 11
H_MAX    = 9                         # highest harmonic counted in THD
OUTLIER_LSB = 3                      # a conversion this far (LSB) from the best-fit line is a glitch: reported as an error and left out of the INL/DNL numbers
SMOOTH_LSB = 8                       # width of the moving average (in LSB of input) that removes quantization error from the INL estimate

SPECS = dict(
    inl_lsb_max      = 0.5,      # smoothed INL, best-fit line (LSB); judged together with the sweep's own floor
    enob_min         = 6.5,
    thd_db_max       = -45.0,    # dBc
    sfdr_db_min      = 45.0,     # dBc
    offset_lsb_max   = None,     # None = report only
    gain_err_pct_max = None,
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3, 'figure.constrained_layout.use': True})

NOTES = []
RESULTS = []
def note(msg):
    if msg not in NOTES:
        NOTES.append(msg)
    print('NOTE:', msg)
def add(name, val, spec, ok):
    RESULTS.append((name, val, spec, 'PASS' if ok else 'FAIL'))
def info(name, val, why=''):
    RESULTS.append((name, val, why, 'INFO'))
def limit(name, val, fmt, lim, kind='<=', unit=''):
    txt = fmt.format(val) if np.isfinite(val) else 'n/a'
    if lim is None:
        info(name, txt, 'no limit set'); return
    ok = np.isfinite(val) and (val <= lim if kind == '<=' else val >= lim)
    add(name, txt, f'{kind} {lim:g} {unit}'.strip(), ok)

def load_table(path, ncols):
    rows = []
    for line in Path(path).read_text().splitlines():
        tok = line.replace(',', ' ').split()
        try:
            r = [float(t) for t in tok]
        except ValueError:
            continue
        if len(r) >= ncols:
            rows.append(r[:ncols])
    if not rows:
        raise ValueError(f'no numeric rows in {path}')
    return np.array(rows)

NCODE = 2 ** N_BITS
V_FS_CIRCUIT = R_OHM * I_FS
G_IDEAL = NCODE / V_FS_CIRCUIT               # codes per volt of V+ - V-
IDD_UA = []
EOC_BAD = 0


def load_dout(path, ncols):
    """last column (code_dout) of a sweep file written by adc_control_all.spice, in run order; None for older files without it"""
    try:
        X = load_table(path, ncols)
    except ValueError:
        return None
    X = X[np.argsort(X[:, 0])]
    return X[:, ncols - 1]


def dout_report(label, code, code_dout):
    """compare the d_out register (read after the last edge) with the SAR-register code: lost bits and what preceded them"""
    code = np.asarray(code).astype(int)
    co = np.asarray(code_dout).astype(int)
    prev = np.r_[0, code[:-1]]
    xor = code ^ co
    nbad = int(np.count_nonzero(xor))
    if nbad == 0:
        info(f'd_out vs SAR register ({label})', f'agree in all {len(code)} conversions', '')
        print(f'd_out vs SAR register ({label}): agree in all {len(code)} conversions')
        return
    parts = []
    for b in range(N_BITS - 1, -1, -1):
        m = 1 << b
        lost = int(np.sum((xor & m > 0) & (code & m > 0)))
        extra = int(np.sum((xor & m > 0) & (code & m == 0)))
        if lost or extra:
            rise = (code & m > 0) & (prev & m == 0)
            high = (code & m > 0) & (prev & m > 0)
            txt = f'bit {m}: lost in {int(np.sum((xor & m > 0) & rise))} of {int(rise.sum())} rises (previous conversion low), in {int(np.sum((xor & m > 0) & high))} of {int(high.sum())} cases where the previous one was high'
            if extra:
                txt += f', set wrongly in {extra}'
            parts.append(txt)
    msg = f'd_out differs from the SAR register in {nbad} of {len(code)} conversions; ' + '; '.join(parts)
    print(f'd_out vs SAR register ({label}): {msg}')
    info(f'd_out vs SAR register ({label})', f'{nbad} of {len(code)} differ', '; '.join(parts) + '. The SAR register code is used; this is the d_cosim output path, not the converter.')

## 1. Static transfer

In [ ]:
STATIC = None


def _transitions(vd, code):
    T = {}
    for k in range(int(code.min()) + 1, int(code.max()) + 1):
        idx = int(np.argmax(code >= k))
        if idx > 0 and code[idx] >= k:
            T[k] = 0.5 * (vd[idx - 1] + vd[idx])
    ks = np.array(sorted(T))
    Tv = np.array([T[k] for k in ks])
    ta, tb = np.polyfit(ks, Tv, 1)
    return ks, Tv, ta, tb


def _linearity(vd, code, g, c, w):
    resid = code + 0.5 - (g * vd + c)
    inl_s = np.convolve(resid, np.ones(w) / w, mode='valid')
    ks, Tv, ta, tb = _transitions(vd, code)
    return dict(inl_s=inl_s, ks=ks, ta=ta, tb=tb,
                inl_t=(Tv - (ta * ks + tb)) / ta, dnl_t=np.diff(Tv) / ta - 1.0, dnl_k=ks[:-1])


def _verdict(name, val, floor, lim):
    if lim is None:
        info(name, f'{val:.3f} LSB', 'no limit set'); return
    txt = f'{val:.3f} LSB (sweep floor {floor:.2f})'
    if val + floor <= lim:
        add(name, txt, f'<= {lim:g} LSB', True)
    elif val - floor > lim:
        add(name, txt, f'<= {lim:g} LSB', False)
    else:
        info(name, txt, f'true value is between {max(val - floor, 0):.2f} and {val + floor:.2f} LSB, which straddles the {lim:g} LSB limit: use a finer sweep')


p = Path(DATA_DIR) / RAMP_FILE
if p.exists():
    M = load_table(p, 4)
    M = M[np.argsort(M[:, 0])]
    vd, code, eoc, idd = M[:, 0], M[:, 1], M[:, 2], M[:, 3]
    if np.any(eoc < 0.5):
        EOC_BAD += int(np.sum(eoc < 0.5))
        note(f'{RAMP_FILE}: {int(np.sum(eoc < 0.5))} of {len(eoc)} conversions had not finished at the sample time (eoc low); they are left out of the static analysis.')
    DOUT_X = load_dout(p, 5)
    if DOUT_X is not None:
        dout_report('ramp A', code, DOUT_X)
    glitch_ab = np.zeros(len(vd), bool)
    TWO, n_ab_bad, n_ab_close = False, 0, 0
    pB = Path(DATA_DIR) / RAMP_FILE_B
    if pB.exists():
        MB = load_table(pB, 4)
        MB = MB[np.argsort(MB[:, 0])]
        if len(MB) == len(M) and np.allclose(MB[:, 0], M[:, 0], atol=1e-7):
            TWO = True
            codeB, eocB = MB[:, 1], MB[:, 2]
            DOUT_XB = load_dout(pB, 5)
            if DOUT_XB is not None:
                dout_report('ramp B', codeB, DOUT_XB)
            dcode = np.abs(code - codeB)
            glitch_ab = (dcode > 1) | ((eoc > 0.5) != (eocB > 0.5))
            n_ab_close = int(np.sum((dcode == 1) & ~glitch_ab))
            n_ab_bad = int(glitch_ab.sum())
            if n_ab_bad:
                note('the two runs of the sweep disagree by more than 1 code at ' +
                     '; '.join(f'vd = {1e3 * v:+.2f} mV: {int(a)} vs {int(b)}' for v, a, b in zip(vd[glitch_ab], code[glitch_ab], codeB[glitch_ab])) +
                     '. Those points are left out of INL/DNL.')
        else:
            note(f'{RAMP_FILE_B} does not have the same sweep points as {RAMP_FILE}: ignored.')
    else:
        note(f'only one run of the sweep ({RAMP_FILE_B} not found): a glitch smaller than {OUTLIER_LSB:g} LSB cannot be told from real INL.')
    keep = (eoc > 0.5) & ~glitch_ab
    vd, code, idd = vd[keep], code[keep], idd[keep]
    IDD_UA.extend(idd)

    OUT = np.zeros(len(vd), bool)
    for _ in range(5):
        g, c = np.polyfit(vd[~OUT], code[~OUT] + 0.5, 1)
        new = np.abs(code + 0.5 - (g * vd + c)) > OUTLIER_LSB
        if new.sum() > 0.2 * len(vd):
            note(f'more than 20 % of the conversions are over {OUTLIER_LSB:g} LSB from a straight line: nothing is set aside, look at the transfer plot')
            OUT[:] = False
            g, c = np.polyfit(vd, code + 0.5, 1)
            break
        if np.array_equal(new, OUT):
            break
        OUT = new
    vd_all, code_all = vd, code
    vd_bad, code_bad = vd[OUT], code[OUT]
    vd, code = vd[~OUT], code[~OUT]
    g, c = np.polyfit(vd, code + 0.5, 1)
    exp_bad = g * vd_bad + c - 0.5
    if OUT.any():
        note(f'{int(OUT.sum())} conversion(s) are more than {OUTLIER_LSB:g} LSB from the line and are left out of INL/DNL: ' +
             '; '.join(f'vd = {1e3 * v:+.2f} mV gave {int(k)} (line says {e:.0f})' for v, k, e in zip(vd_bad, code_bad, exp_bad)))
    step = float(np.median(np.diff(vd)))
    w = max(3, int(round(SMOOTH_LSB / (step * g))))      # points; about 9 at one point per 0.86 LSB, more for a finer sweep
    lsb_v = 1.0 / g
    cmin, cmax = int(code.min()), int(code.max())
    gain_err = 100 * (g / G_IDEAL - 1)
    offset_lsb = float(c - NCODE / 2)
    missing = [k for k in range(cmin, cmax + 1) if not np.any(code == k)]
    mono_viol = int(np.sum(np.diff(code) < 0))

    meas = _linearity(vd, code, g, c, w)
    vd_s = np.convolve(vd, np.ones(w) / w, mode='valid')
    # the same sweep points through a perfect ADC with the same gain and offset: this is the floor of the method
    code_i = np.clip(np.floor(g * vd + c), cmin, cmax)
    ideal = _linearity(vd, code_i, g, c, w)

    inl_max = float(np.max(np.abs(meas['inl_s'])))
    floor_s = float(np.max(np.abs(ideal['inl_s'])))
    inl_t_max = float(np.max(np.abs(meas['inl_t'])))
    floor_t = float(np.max(np.abs(ideal['inl_t'])))
    dnl_t, dnl_k = meas['dnl_t'], meas['dnl_k']
    res_lsb = step / meas['ta']
    ta, tb, ks = meas['ta'], meas['tb'], meas['ks']

    STATIC = dict(vd_bad=vd_bad, code_bad=code_bad, vd=vd, code=code, g=g, c=c, inl_s=meas['inl_s'], vd_s=vd_s, ks=ks, inl_t=meas['inl_t'], dnl_t=dnl_t, dnl_k=dnl_k,
                  floor_s=floor_s, floor_t=floor_t)
    print(f'{len(vd_all)} conversions ({len(vd_bad)} set aside as glitches), vd {1e3 * vd.min():+.0f} .. {1e3 * vd.max():+.0f} mV (step {1e3 * step:.2f} mV = {res_lsb:.2f} LSB), codes {cmin} .. {cmax}')
    print(f'LSB from the fit: {1e3 * lsb_v:.3f} mV (ideal {1e3 * V_FS_CIRCUIT / NCODE:.3f} mV);  gain error {gain_err:+.2f} %;  offset {offset_lsb:+.2f} LSB')
    print(f'smoothed INL ({w}-point average): {inl_max:.3f} LSB peak;  a perfect ADC on this sweep reads {floor_s:.3f}')
    print(f'transition-based INL: {inl_t_max:.2f} LSB peak;  a perfect ADC on this sweep reads {floor_t:.2f}')
    print(f'transition-based DNL: {dnl_t.min():+.2f} .. {dnl_t.max():+.2f} LSB;  a perfect ADC on this sweep reads {ideal["dnl_t"].min():+.2f} .. {ideal["dnl_t"].max():+.2f}')
    print(f'missing codes between {cmin} and {cmax}: {missing if missing else "none"};  non-monotonic steps: {mono_viol}')
    if res_lsb > 0.5:
        beat = 1 / abs(1 - res_lsb) if abs(1 - res_lsb) > 1e-3 else float('inf')
        note(f'The sweep step is {res_lsb:.2f} LSB, so even a perfect ADC reads {floor_t:.2f} LSB of transition-based INL and DNL up to {ideal["dnl_t"].max():+.2f} here, with a pattern that repeats about every {beat:.1f} codes. That is the sweep beating against the code width, not the circuit. For INL/DNL use 4 or more points per LSB (npts of 513 or more).')

    fig, axs = plt.subplots(1, 3, figsize=(13, 3.6))
    axs[0].plot(1e3 * vd, code, '.', ms=3, label='measured')
    if len(vd_bad):
        axs[0].plot(1e3 * vd_bad, code_bad, 'rx', ms=7, label=f'{len(vd_bad)} glitch(es)')
    axs[0].plot(1e3 * vd, g * vd + c - 0.5, lw=0.8, label='best-fit line')
    axs[0].set_xlabel('V+ - V- (mV)'); axs[0].set_ylabel('code'); axs[0].legend(); axs[0].set_title('transfer')
    axs[1].plot(1e3 * (ta * ks + tb), meas['inl_t'], '.', ms=3, c='C1', label='measured, transitions')
    axs[1].plot(1e3 * vd_s, meas['inl_s'], c='C0', label=f'measured, smoothed ({w} pt)')
    axs[1].plot(1e3 * (ideal['ta'] * ideal['ks'] + ideal['tb']), ideal['inl_t'], 'x', ms=3, c='k', label='perfect ADC, transitions')
    axs[1].plot(1e3 * vd_s, ideal['inl_s'], 'k--', lw=0.9, label='perfect ADC, smoothed')
    axs[1].axhline(SPECS['inl_lsb_max'], c='0.5', ls=':'); axs[1].axhline(-SPECS['inl_lsb_max'], c='0.5', ls=':')
    axs[1].set_xlabel('V+ - V- (mV)'); axs[1].set_ylabel('LSB'); axs[1].legend(fontsize=7); axs[1].set_title('INL')
    axs[2].bar(dnl_k, dnl_t, width=0.8, label='measured')
    axs[2].plot(ideal['dnl_k'], ideal['dnl_t'], 'k_', ms=5, label='perfect ADC')
    axs[2].axhline(-1, c='r', ls=':'); axs[2].legend(fontsize=7)
    axs[2].set_xlabel('code'); axs[2].set_ylabel('LSB'); axs[2].set_title(f'DNL (sweep step {res_lsb:.2f} LSB)')
    plt.show()

    add('conversion errors', f'{len(vd_bad)} beyond +/-{OUTLIER_LSB:g} LSB of the line' + (f' (of {len(vd_all)})' if len(vd_bad) else ''), '== 0', len(vd_bad) == 0)
    if TWO:
        add('two runs agree', f'{n_ab_bad} differ by more than 1 code, {n_ab_close} by 1 code (of {len(M)})', '0 differ by more than 1 code', n_ab_bad == 0)
    else:
        info('two runs agree', 'single run only', 'run both ramp sections of adc_control_all.spice to catch glitches')
    _verdict('INL, smoothed best-fit', inl_max, floor_s, SPECS['inl_lsb_max'])
    info('INL, from transitions', f'{inl_t_max:.2f} LSB', f'a perfect ADC on this sweep reads {floor_t:.2f} LSB')
    if res_lsb < 1:
        add('no missing codes', f'{len(missing)} missing', '== 0', len(missing) == 0)
    else:
        info('missing codes', f'{len(missing)} skipped', 'the sweep step is not smaller than an LSB, so skipped codes may be sampling: use more points')
    add('monotonic', f'{mono_viol} reversals', '== 0', mono_viol == 0)
    info('DNL, from transitions', f'{dnl_t.min():+.2f} .. {dnl_t.max():+.2f} LSB', f'a perfect ADC on this sweep reads {ideal["dnl_t"].min():+.2f} .. {ideal["dnl_t"].max():+.2f}: only the part beyond that is the circuit')
    limit('gain error', abs(gain_err), '{:.2f} %', SPECS['gain_err_pct_max'], '<=', '%')
    limit('offset', abs(offset_lsb), '{:.2f} LSB', SPECS['offset_lsb_max'], '<=', 'LSB')
else:
    note(f'{RAMP_FILE} not found in {DATA_DIR}')

## 2. Sine: SINAD, ENOB, THD

In [ ]:
SINE = None
SINE_TOL = 1.5     # codes: a sine sample this far from the ramp's code at the same input is a conversion error
p = Path(DATA_DIR) / SINE_FILE
if p.exists():
    M = load_table(p, 5)
    M = M[np.argsort(M[:, 0])]
    nn, vd, code, eoc, idd = M.T
    # each sine sample is a static conversion, so the ramp says what code it should give
    EXP = np.interp(vd, STATIC['vd'], STATIC['code']) if STATIC is not None else None
    SINE_TWO, n_sb_bad, n_sb_close = False, 0, 0
    DOUT_S = load_dout(p, 6)
    if DOUT_S is not None:
        dout_report('sine A', code, DOUT_S)
    code_a, eoc_a, code_b, eoc_b = code.copy(), eoc.copy(), None, None
    pB = Path(DATA_DIR) / SINE_FILE_B
    if pB.exists():
        MB = load_table(pB, 5)
        MB = MB[np.argsort(MB[:, 0])]
        if len(MB) == len(M) and np.allclose(MB[:, 1], M[:, 1], atol=1e-7):
            SINE_TWO = True
            codeB, eocB = MB[:, 2], MB[:, 3]
            code_b, eoc_b = codeB.copy(), eocB.copy()
            DOUT_SB = load_dout(pB, 6)
            if DOUT_SB is not None:
                dout_report('sine B', codeB, DOUT_SB)
            dcode = np.abs(code - codeB)
            bad = (dcode > 1) | ((eoc > 0.5) != (eocB > 0.5))
            n_sb_close = int(np.sum((dcode == 1) & ~bad))
            n_sb_bad = int(bad.sum())
            if n_sb_bad:
                if EXP is not None:
                    ref, ref_txt = EXP, 'the ramp gives at the same input'
                else:
                    th = 2 * np.pi * SINE_M * np.arange(len(code)) / len(code)
                    X = np.column_stack([np.sin(th), np.cos(th), np.ones_like(th)])
                    ref, ref_txt = X @ np.linalg.lstsq(X[~bad], code[~bad], rcond=None)[0], 'a sine fitted to the other samples gives'
                costA = np.abs(code - ref) + 1e3 * (eoc < 0.5)
                costB = np.abs(codeB - ref) + 1e3 * (eocB < 0.5)
                pickB = bad & (costB < costA)
                note('the two runs of the sine disagree by more than 1 code at sample(s) ' +
                     ', '.join(f'{int(k)} (A {int(a)}, B {int(b)})' for k, a, b in zip(nn[bad], code[bad], codeB[bad])) +
                     f'; for each, the run closer to what {ref_txt} is used.')
                code = np.where(pickB, codeB, code)
                eoc = np.where(pickB, eocB, eoc)
        else:
            note(f'{SINE_FILE_B} does not have the same samples as {SINE_FILE}: ignored.')
    else:
        note(f'only one run of the sine ({SINE_FILE_B} not found): a glitch cannot be told from converter error and will spoil ENOB and THD.')
    if np.any(eoc < 0.5):
        EOC_BAD += int(np.sum(eoc < 0.5))
        note(f'{SINE_FILE}: {int(np.sum(eoc < 0.5))} conversions had not finished at the sample time; the spectrum is not valid.')
    IDD_UA.extend(idd[eoc > 0.5])
    N = len(code)
    if N != SINE_N:
        note(f'{SINE_FILE} has {N} samples but SINE_N is {SINE_N}; using {N}. Coherence needs SINE_M cycles in the record.')
    Mc = SINE_M

    def spectrum_metrics(x, Mc, hmax):
        x = np.asarray(x, float) - np.mean(x)
        n = len(x)
        X = np.fft.rfft(x) / n
        P = np.abs(X) ** 2
        P[1:(n + 1) // 2] *= 2
        P[0] = 0.0
        pf = P[Mc]
        hb = []
        for h in range(2, hmax + 1):
            b = (h * Mc) % n
            b = n - b if b > n // 2 else b
            if b not in (0, Mc) and b not in hb:
                hb.append(b)
        ph = float(np.sum(P[hb])) if hb else 0.0
        other = P.copy(); other[Mc] = 0.0
        sinad = 10 * np.log10(pf / np.sum(other))
        noise = other.copy(); noise[hb] = 0.0
        snr = 10 * np.log10(pf / np.sum(noise))
        thd = 10 * np.log10(max(ph, 1e-30) / pf)
        sfdr = 10 * np.log10(pf / np.max(other))
        app = 2 * np.sqrt(2 * pf)
        enob = (sinad - 1.76 + 20 * np.log10(NCODE / app)) / 6.02
        return dict(P=P, hb=hb, sinad=sinad, snr=snr, thd=thd, sfdr=sfdr, app=app, enob=enob)

    m = spectrum_metrics(code, Mc, H_MAX)
    g, c = np.polyfit(vd, code + 0.5, 1)
    ideal = np.clip(np.floor(g * vd + c), 0, NCODE - 1)
    mi = spectrum_metrics(ideal, Mc, H_MAX)
    sinad_theory = 6.02 * N_BITS + 1.76 + 20 * np.log10(m['app'] / NCODE)

    err = np.zeros(N, bool)
    code_fix, mf = code, m
    if EXP is not None:
        def _errs(c, e):
            return (np.abs(c - EXP) > SINE_TOL) | (e < 0.5)
        err = _errs(code, eoc)
        errA = _errs(code_a, eoc_a)
        errB = _errs(code_b, eoc_b) if code_b is not None else np.zeros(N, bool)
        rows = []
        for run, cc, er in (('A', code_a, errA), ('B', code_b, errB)):
            if cc is None:
                continue
            for i in np.flatnonzero(er):
                rows.append(dict(run=run, sample=int(nn[i]), vd_mV=round(1e3 * vd[i], 1), got=int(cc[i]), ramp_says=round(float(EXP[i]), 1),
                                 diff=int(round(cc[i] - EXP[i])), prev_sample_ramp_says=round(float(EXP[i - 1]), 1) if i > 0 else np.nan))
        print(f'sine vs ramp (each sine sample is a static conversion, so it should match the ramp within {SINE_TOL:g} codes): '
              f'run A {int(errA.sum())} of {N} off' + (f', run B {int(errB.sum())} off, both runs off at the same sample: {int((errA & errB).sum())}' if code_b is not None else '') +
              f'; {int(err.sum())} remain after merging the runs')
        if rows:
            from collections import Counter
            print('  got - ramp, all bad conversions:', dict(sorted(Counter(r['diff'] for r in rows).items())))
            display(pd.DataFrame(rows).head(60))
            code_fix = np.where(err, np.round(EXP), code)
            mf = spectrum_metrics(code_fix, Mc, H_MAX)
            print(f'with the {int(err.sum())} bad conversion(s) replaced by the ramp\'s code at the same input: SINAD {mf["sinad"]:.2f} dB   THD {mf["thd"]:.2f} dBc   SFDR {mf["sfdr"]:.2f} dBc   ENOB {mf["enob"]:.2f}'
                  '   <- an estimate of the converter with the controller output right, not a measurement')
            note(f'{int(err.sum())} sine conversion(s) differ from what the ramp gives at the same input. A converter error would show in the ramp too, so these are conversion glitches (a lost or wrong output bit), not linearity. '
                 'ENOB, THD and SFDR below are not graded; the check "sine matches ramp" fails instead.')
    SINE = dict(m=m, mi=mi, code=code, ideal=ideal, vd=vd, m_eff=mf)
    print(f'{N} samples, {Mc} cycles, signal {m["app"]:.1f} codes peak-to-peak ({100 * m["app"] / NCODE:.0f} % of full scale)')
    print(f'measured:            SINAD {m["sinad"]:.2f} dB   SNR {m["snr"]:.2f} dB   THD {m["thd"]:.2f} dBc   SFDR {m["sfdr"]:.2f} dBc   ENOB {m["enob"]:.2f}')
    print(f'ideal quantizer, same samples, same gain and offset: SINAD {mi["sinad"]:.2f} dB   THD {mi["thd"]:.2f} dBc   ENOB {mi["enob"]:.2f}')
    print(f'theory for a {N_BITS}-bit ideal ADC at this amplitude: SINAD {sinad_theory:.2f} dB')

    fn = np.arange(len(m['P']))
    fig, axs = plt.subplots(1, 2, figsize=(12, 3.8))
    db = 10 * np.log10(np.maximum(m['P'], 1e-30) / m['P'][Mc])
    axs[0].vlines(fn[1:], -120, db[1:], lw=1.0)
    axs[0].plot(Mc, 0, 'o', c='C3'); axs[0].plot(m['hb'], db[m['hb']], 'o', c='C1', ms=4)
    axs[0].set_ylim(-100, 5); axs[0].set_xlabel('bin'); axs[0].set_ylabel('dBc')
    axs[0].set_title(f'spectrum: ENOB {m["enob"]:.2f}, THD {m["thd"]:.1f} dBc, SFDR {m["sfdr"]:.1f} dBc')
    k = np.arange(N)
    axs[1].step(k, code, where='mid', label='measured'); axs[1].plot(k, g * vd + c - 0.5, lw=0.8, label='ideal (continuous)')
    axs[1].plot(k, code - ideal, '.', ms=3, label='measured - ideal quantizer')
    if err.any():
        axs[1].plot(k[err], code[err], 'rx', ms=7, label=f'{int(err.sum())} not matching the ramp')
    axs[1].set_xlabel('sample'); axs[1].set_ylabel('code'); axs[1].set_title('records')
    axs[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2, fontsize=8)
    plt.show()

    if SINE_TWO:
        add('two runs agree (sine)', f'{n_sb_bad} differ by more than 1 code, {n_sb_close} by 1 code (of {len(code)})', '0 differ by more than 1 code', n_sb_bad == 0)
    else:
        info('two runs agree (sine)', 'single run only', 'run both sine sections of adc_control_all.spice to catch glitches')
    if EXP is not None:
        add('sine matches ramp', f'{int(err.sum())} of {N} more than {SINE_TOL:g} codes off', '== 0', not err.any())
    if err.any():
        for nm, key, fmt in (('ENOB', 'enob', '{:.2f} bit'), ('THD', 'thd', '{:.1f} dBc'), ('SFDR', 'sfdr', '{:.1f} dBc')):
            info(nm, (fmt + ' measured, ' + fmt + ' with errors replaced').format(m[key], mf[key]), 'not graded: the sine has conversion glitches')
    else:
        limit('ENOB', m['enob'], '{:.2f} bit', SPECS['enob_min'], '>=', 'bit')
        limit('THD', m['thd'], '{:.1f} dBc', SPECS['thd_db_max'], '<=', 'dBc')
        limit('SFDR', m['sfdr'], '{:.1f} dBc', SPECS['sfdr_db_min'], '>=', 'dBc')
    info('SINAD / SNR', f'{m["sinad"]:.1f} / {m["snr"]:.1f} dB', f'ideal quantizer on the same samples: {mi["sinad"]:.1f} dB')
else:
    note(f'{SINE_FILE} not found in {DATA_DIR}')

## Summary

In [ ]:
if IDD_UA:
    idd = float(np.mean(IDD_UA))
    pw = idd * 1e-6 * VDD
    info('supply current / power', f'{idd:.1f} uA / {1e6 * pw:.1f} uW', 'excludes the ideal bias sources')
    if SINE is not None:
        fom = pw / (F_CONV * 2 ** SINE['m_eff']['enob'])
        info('Walden FoM', f'{1e15 * fom:.0f} fJ/conv-step', f'at {F_CONV / 1e6:g} MS/s and the measured ENOB')
if EOC_BAD:
    add('all conversions finished', f'{EOC_BAD} unfinished', '== 0', False)
SUMMARY = pd.DataFrame(RESULTS, columns=['check', 'value', 'spec', 'result'])
display(SUMMARY)
print(SUMMARY['result'].value_counts().to_dict())
for n in NOTES:
    print('NOTE:', n)